# 01 — Data Exploration
**Wake Word Detection & Intent Classification System**

This notebook explores the synthetic training data:
- MFCC spectrograms for positive/negative wake-word samples
- Intent class distribution
- Audio waveform visualisation

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from src.audio.feature_extraction import MFCCExtractor, MFCCConfig
from src.audio.preprocessor import AudioPreprocessor
from src.audio.audio_capture import AudioStreamer
from src.training.dataset import WakeWordDataset, IntentDataset, INTENT_LABELS

plt.style.use('dark_background')
print('Setup complete.')

In [ ]:
# ── Generate a few samples ────────────────────────────────────────────────────
SR = 16_000
extractor = MFCCExtractor(MFCCConfig(n_mfcc=40, max_frames=98, delta_order=0))

t = np.linspace(0, 1.0, SR, endpoint=False)
positive_audio = (0.5 * np.sin(2 * np.pi * 1000 * t)).astype(np.float32)
negative_audio = np.random.randn(SR).astype(np.float32) * 0.3

mfcc_pos = extractor.extract(positive_audio)
mfcc_neg = extractor.extract(negative_audio)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('MFCC Spectrograms — Wake Word vs. Background', fontsize=14)

axes[0, 0].plot(t, positive_audio, color='cyan', lw=0.5)
axes[0, 0].set_title('Wake Word (1 kHz tone)')
axes[0, 0].set_xlabel('Time (s)')

axes[0, 1].plot(t, negative_audio, color='orange', lw=0.5)
axes[0, 1].set_title('Background Noise')
axes[0, 1].set_xlabel('Time (s)')

im1 = axes[1, 0].imshow(mfcc_pos, aspect='auto', origin='lower', cmap='magma')
axes[1, 0].set_title('MFCC — Wake Word')
axes[1, 0].set_ylabel('MFCC Coefficient')
axes[1, 0].set_xlabel('Frame')
plt.colorbar(im1, ax=axes[1, 0])

im2 = axes[1, 1].imshow(mfcc_neg, aspect='auto', origin='lower', cmap='magma')
axes[1, 1].set_title('MFCC — Background Noise')
axes[1, 1].set_xlabel('Frame')
plt.colorbar(im2, ax=axes[1, 1])

plt.tight_layout()
plt.show()

In [ ]:
# ── Intent class distribution ─────────────────────────────────────────────────
intent_ds_gen = IntentDataset(n_per_class=50, seed=42)
hf_dataset = intent_ds_gen.generate()

import pandas as pd
df = hf_dataset.to_pandas()
counts = df['label'].value_counts().sort_index()
label_names = [INTENT_LABELS[i] for i in counts.index]

fig, ax = plt.subplots(figsize=(14, 5))
colors = list(mcolors.TABLEAU_COLORS.values()) * 3
ax.bar(label_names, counts.values, color=colors[:len(label_names)])
ax.set_xlabel('Intent Class', fontsize=12)
ax.set_ylabel('Sample Count', fontsize=12)
ax.set_title('Intent Class Distribution (50 samples/class)', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

print(f'Total samples: {len(df)}')
print(f'Unique intents: {df["label"].nunique()}')
print('\nSample utterances:')
for i in range(5):
    row = df.iloc[i]
    print(f'  [{INTENT_LABELS[row["label"]]}] {row["text"]}')